In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [5]:
len(documents)

72

In [11]:
doc = documents[0]
doc
print(doc["content"])
print(doc["filename"])

# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack](https://alexeyondata.substack.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](https://x.com/Al_Grigor)

## LLMs

An LLM (Large Language Model) is a neural network trained on massive
amounts of text. Given a prompt, it generates a continuation - a
plausible next piece of text.

Think of your phone. When you type "how are" in WhatsApp, it suggests
"you" as the next word. "How are you" is the most common continuation.
Your phone uses a simple language model for that. It predicts 

In [12]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [13]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [14]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [15]:
import json

user_prompt = json.dumps(doc)

user_prompt

'{"content": "# Introduction\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn this module, we\'ll build a working Retrieval-Augmented\\nGeneration (RAG) system from scratch, step by step.\\n\\nWe write everything in plain Python. We build a small search index by\\nhand and call the LLM ourselves. I want you to see every piece first.\\nThat way you know what a framework does for you before you reach for\\none.\\n\\nPlaces where you can find me:\\n\\n- [My substack](https://alexeyondata.substack.com/)\\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\\n- [X](https://x.com/Al_Grigor)\\n\\n## LLMs\\n\\nAn LLM (Large Language Model) is a neural network trained on massive\\namounts of text. Given a prompt, it generates a continuation - a\\nplausible next piece of text.\\n\\nThink of your phone. When you type \\"how are\\" in WhatsApp, it suggests\\n\\"you\\" as the next word. \\"How are you\\" is the most common

In [16]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [17]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [18]:
result = response.output_parsed

print(result)

questions=['What is RAG, and why does it help when an LLM doesn’t know your documents or current facts?', 'How does this course define an LLM, and how is it different from a simple next-word predictor on a phone?', 'Why does the lesson say LLMs are treated like black boxes instead of studying their internals?', 'What are the main parts of the first half of this RAG module, and what will be built step by step?', 'What changes in the second part of the module when the pipeline becomes agentic?']


In [19]:
print(result.questions)

['What is RAG, and why does it help when an LLM doesn’t know your documents or current facts?', 'How does this course define an LLM, and how is it different from a simple next-word predictor on a phone?', 'Why does the lesson say LLMs are treated like black boxes instead of studying their internals?', 'What are the main parts of the first half of this RAG module, and what will be built step by step?', 'What changes in the second part of the module when the pipeline becomes agentic?']


In [20]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

["What is a Retrieval-Augmented Generation system, and why does it help when the model doesn't know the answer on its own?", 'Why does the course build the RAG setup in plain Python instead of using a framework right away?', 'What are some main limits of large language models that RAG is meant to fix?', 'What does the FAQ agent in this module do, and what kind of question should it be able to answer?', 'What will be covered in the first part of the module versus the second part?']


In [21]:
print(usage)

ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=116, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1136)


In [22]:
from evaluation_utils import calc_price

cost = calc_price(usage)

cost

{'input_cost': 0.0007650000000000001,
 'output_cost': 0.000522,
 'total_cost': 0.001287}

In [23]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["filename"]
    })

records

[{'question': "What is a Retrieval-Augmented Generation system, and why does it help when the model doesn't know the answer on its own?",
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does the course build the RAG setup in plain Python instead of using a framework right away?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are some main limits of large language models that RAG is meant to fix?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What does the FAQ agent in this module do, and what kind of question should it be able to answer?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will be covered in the first part of the module versus the second part?',
  'document': '01-agentic-rag/lessons/01-intro.md'}]

In [24]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [25]:
generate_ground_truth(doc)

([{'question': 'What is a Retrieval-Augmented Generation system, and how does it help with answers the model didn’t learn during training?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'Why does this course build the RAG pipeline in plain Python instead of using a framework right away?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What are the main limits of large language models that RAG is meant to work around?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What kind of project will be built in this module, and what question-answering example does it use?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What topics are covered in the first part of the module, and what changes in the second part?',
   'document': '01-agentic-rag/lessons/01-intro.md'}],
 ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=119, outp

In [26]:
# Question 1
target_filenames = {
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
}

generated_records = []

for doc in documents:
    if doc["filename"] in target_filenames:
        ground_truth, usage = generate_ground_truth(doc)
        generated_records.extend(ground_truth)
        print(doc["filename"], usage)

generated_records

01-agentic-rag/lessons/01-intro.md ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=114, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1134)
01-agentic-rag/lessons/02-environment.md ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1398)
01-agentic-rag/lessons/03-rag.md ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=116, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1869)


[{'question': 'What is a Retrieval-Augmented Generation system, and how does it help with answers the model might not know on its own?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course build the RAG pipeline in plain Python instead of using a framework right away?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kinds of limits do large language models have in this lesson, like missing recent info or not seeing private data?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What example project does this module build to show RAG in practice?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will be covered in the first part versus the second part of this module?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What do I need installed before starting this module, besides Python and Jupyter?',
  'document': '01-agentic-rag/lessons/02-environment.md'},
 {'q

In [1]:
import pandas as pd

# Load CSV into a DataFrame
df = pd.read_csv("data/ground-truth.csv")

# Convert DataFrame rows into a list of dictionaries
ground_truth = df.to_dict(orient="records")
len(ground_truth)

360

In [6]:
from gitsource import chunk_documents


chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [17]:
from minsearch import Index,VectorSearch
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

In [11]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

In [12]:
q = ground_truth[0]["question"]
print(q)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


In [13]:
text_results = text_search(q)
text_results[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

In [15]:
from embedder import Embedder

embed = Embedder()

2026-07-11 11:14:56.642519515 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [16]:
texts = [chunk["content"] for chunk in chunks]
X = embed.encode_batch(texts)

In [18]:
X.shape

(295, 384)

In [19]:
vector_index = VectorSearch()
vector_index.fit(X, chunks)

In [20]:
q = ground_truth[0]["question"]

In [26]:
def vector_search(query, num_results=10):
    query_vector = embed.encode(query)
    return vector_index.search(query_vector,
                         num_results=num_results)

In [27]:
vector_results = vector_search(q,5)
vector_results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [28]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [29]:
def hybrid_search(query, k=60,num_results=10):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)

In [30]:
from tqdm.auto import tqdm

def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

In [31]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance_total.append(compute_relevance(q, search_function))
    return relevance_total

In [32]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt = cnt + 1
    return cnt / len(relevance)

In [33]:
def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break
    return total_score / len(relevance)

In [34]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [35]:
result_text = evaluate(ground_truth, text_search)
result_text

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [36]:
result_vector = evaluate(ground_truth, vector_search)
result_vector

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8361111111111111, 'mrr': 0.5646472663139328}

In [37]:
results_by_k = {}
for k in [1, 50, 100, 200]:
    result = evaluate(ground_truth, lambda query, k=k: hybrid_search(query, k=k))
    results_by_k[k] = result
    print(f"k={k}: {result}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: {'hit_rate': 0.9083333333333333, 'mrr': 0.6576917989417991}


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: {'hit_rate': 0.9083333333333333, 'mrr': 0.6479872134038801}


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: {'hit_rate': 0.9083333333333333, 'mrr': 0.6479872134038801}


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: {'hit_rate': 0.9083333333333333, 'mrr': 0.6479872134038801}


In [38]:
best_k = max(results_by_k, key=lambda k: (results_by_k[k]["mrr"], -k))
best_k

1